# W&B Training Loop Integration

Shows how to use `dna-segmentation-benchmark` inside a training loop to log metrics to Weights & Biases at each epoch, then log a full comparison with plots at the end.

Requires: `pip install dna-segmentation-benchmark[wandb]`

In [ ]:
import numpy as np
from dna_segmentation_benchmark import (
    LabelConfig,
    EvalMetrics,
    benchmark_gt_vs_pred_multiple,
    compare_multiple_predictions,
    init_wandb_with_presets,
    log_benchmark_scalars,
    log_benchmark_full,
)

label_config = LabelConfig(
    labels={0: "EXON", 2: "INTRON", 8: "NONCODING"},
    background_label=8,
    coding_label=0,
)
classes = [0, 2]

## Mock validation data

In a real setup, `gt_labels` would be your validation set ground truth and `predict()` would be your model's forward pass.

In [ ]:
rng = np.random.default_rng(42)

# Fixed validation GT (100 sequences of 5k–20k bases)
gt_labels = []
for _ in range(100):
    length = rng.integers(5000, 20000)
    arr = np.full(length, 8, dtype=np.int32)
    pos = rng.integers(50, 200)
    while pos < length - 100:
        arr[pos : pos + rng.integers(50, 300)] = 0
        pos += rng.integers(50, 300)
        arr[pos : pos + rng.integers(200, 1500)] = 2
        pos += rng.integers(200, 1500)
    gt_labels.append(arr)


def predict(gt_labels: list[np.ndarray], epoch: int) -> list[np.ndarray]:
    """Mock model that improves with epoch number."""
    noise = max(0.01, 0.15 - epoch * 0.02)
    preds = []
    for gt in gt_labels:
        pred = gt.copy()
        flip = rng.random(len(gt)) < noise
        pred[flip] = rng.choice([0, 2, 8], size=flip.sum())
        preds.append(pred)
    return preds

## Training loop with per-epoch W&B logging

`init_wandb_with_presets` sets up the W&B run with pre-configured metric groupings so the dashboard is organized automatically. `log_benchmark_scalars` logs lightweight scalar metrics each epoch.

In [ ]:
# Use lightweight metrics during training to keep overhead low
train_metrics = [EvalMetrics.REGION_DISCOVERY, EvalMetrics.NUCLEOTIDE_CLASSIFICATION]

run = init_wandb_with_presets(
    project="dna-benchmark-demo",
    run_name="mock-training-run",
    label_config=label_config,
    classes=classes,
    config={"model": "mock-cnn", "lr": 1e-3, "epochs": 8},
)

for epoch in range(8):
    # ... your training step here ...

    # Evaluate on validation set
    val_preds = predict(gt_labels, epoch)
    val_results = benchmark_gt_vs_pred_multiple(
        gt_labels=gt_labels,
        pred_labels=val_preds,
        label_config=label_config,
        classes=classes,
        metrics=train_metrics,
    )

    # Log scalar metrics (precision, recall, F1) to W&B
    log_benchmark_scalars(val_results, label_config, step=epoch, method_prefix="val")

## Post-training: full diagnostic report

After training, run the full metric suite (including INDEL and boundary analysis) and log everything — figures, IoU distributions, and all scalars — in one call with `log_benchmark_full`.

In [ ]:
full_metrics = [
    EvalMetrics.INDEL,
    EvalMetrics.REGION_DISCOVERY,
    EvalMetrics.BOUNDARY_EXACTNESS,
    EvalMetrics.NUCLEOTIDE_CLASSIFICATION,
]

# Final evaluation with all metrics
final_preds = predict(gt_labels, epoch=7)
final_results = benchmark_gt_vs_pred_multiple(
    gt_labels=gt_labels,
    pred_labels=final_preds,
    label_config=label_config,
    classes=classes,
    metrics=full_metrics,
)

# Generate plots and log everything to W&B
per_method = {"mock-cnn": final_results}
figures = compare_multiple_predictions(
    per_method_benchmark_res=per_method,
    label_config=label_config,
    classes=classes,
    metrics_to_eval=full_metrics,
)

log_benchmark_full(per_method, figures, label_config)
run.finish()